In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## Step 6a: T_PairEnv / T_PairDataset — Pair Structure Task

**Role**: Schapiro (2017) §3.a control task — 4 fixed pairs (AB/CD/EF/GH).  
Each trial: present A as ECin input; B as ECout plus-phase target.  
Pair order is random with no back-to-back repetitions.  
80 inputs/epoch (Schapiro 2017 §3.a).  

**Why pairs?**  
In contrast to the community graph, the pair task has no statistical community structure —  
only pairwise associations. MSP cannot exploit higher-order transition regularities;  
TSP encodes each pair directly. Used to dissociate MSP from TSP contributions.

**Pair structure**:
- Pair 0: items 0→1 (AB)
- Pair 1: items 2→3 (CD)
- Pair 2: items 4→5 (EF)
- Pair 3: items 6→7 (GH)

In [ ]:
# path & directories
import sys
from pathlib import Path

SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
sys.path.insert(0, SRC)

# hyperparameters
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tasks import T_PairEnv, T_PairDataset

# Schapiro (2017) §3.a task parameters
N_PAIRS = 4    # AB / CD / EF / GH
N_STEPS = 800  # 80 inputs/epoch × 10 epochs (Schapiro 2017 §3.a)
N_ITEMS = N_PAIRS * 2
SEED    = 0

In [ ]:
# Instantiate and inspect structure
env = T_PairEnv(n_pairs=N_PAIRS, seed=SEED)

print(f"n_pairs : {env.n_pairs}")
print(f"n_items : {env.n_items}")
print(f"pairs   : {env.pairs}")

In [ ]:
# Run 20 steps; verify (A, B) pairs and no back-to-back same pair
env.reset(seed=0)
transitions = [env.step() for _ in range(20)]

print("step  current  next  pair  valid_pair?")
for i, (cur, nxt) in enumerate(transitions):
    pair_idx = cur // 2
    expected_next = cur + 1          # B always follows A within a pair
    valid = (nxt == expected_next)
    print(f"  {i:2d}      {cur}        {nxt}     {pair_idx}      {valid}")

# Check no back-to-back same pair
pair_sequence = [cur // 2 for cur, _ in transitions]
back_to_back = any(pair_sequence[i] == pair_sequence[i+1] for i in range(len(pair_sequence)-1))
print(f"\nBack-to-back same pair: {back_to_back}  (expect False)")

In [ ]:
# Transition statistics over N_STEPS: P(A→B) should be 1.0 for each pair
# and P(A→anything_else) = 0.0
env.reset(seed=42)
counts = np.zeros((N_ITEMS, N_ITEMS), dtype=int)
for _ in range(N_STEPS):
    cur, nxt = env.step()
    counts[cur, nxt] += 1

print("Transition counts (rows=current item, cols=next item):")
print(counts)
print()
# Items 0,2,4,6 are first-of-pair: all transitions must go to item+1
for a in range(0, N_ITEMS, 2):
    b = a + 1
    total   = counts[a].sum()
    correct = counts[a, b]
    print(f"Item {a} → item {b}: {correct}/{total} ({100*correct/max(total,1):.0f}%)")

In [ ]:
# T_PairDataset: verify shapes, one-hots, and pair label correctness
dataset = T_PairDataset(n_steps=N_STEPS, n_pairs=N_PAIRS, seed=7)

print(f"Dataset length         : {len(dataset)}")
sample = dataset[0]
print(f"Keys                   : {list(sample.keys())}")
print(f"item_onehot shape      : {sample['item_onehot'].shape}")
print(f"target_onehot shape    : {sample['target_onehot'].shape}")

# Verify: for every sample, item is always even (first-of-pair) and next_item = item + 1
all_valid = all(
    (dataset[i]['item'].item() % 2 == 0) and
    (dataset[i]['next_item'].item() == dataset[i]['item'].item() + 1)
    for i in range(len(dataset))
)
print(f"All samples are valid (A→B) pairs: {all_valid}  (expect True)")

# Verify pair balance: each pair should appear roughly equally
pair_counts = torch.zeros(N_PAIRS, dtype=torch.long)
for i in range(len(dataset)):
    pair_counts[dataset[i]['pair']] += 1
print(f"Pair counts: {pair_counts.tolist()}  (expect ~{N_STEPS // N_PAIRS} each)")

In [ ]:
PAIR_COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231']
PAIR_LABELS = [('A', 'B'), ('C', 'D'), ('E', 'F'), ('G', 'H')]
VIZ         = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.set_xlim(-0.5, N_PAIRS * 4 - 0.5)
ax.set_ylim(-0.8, 1.5)
ax.axis('off')
ax.set_title('Pair Structure Task (Schapiro 2017 §3.a)\n'
             'Each trial: A → ECin (input),  B → ECout (plus-phase target)', fontsize=11)

for i, (la, lb) in enumerate(PAIR_LABELS):
    x0    = i * 4 + 0.8
    x1    = i * 4 + 2.7
    color = PAIR_COLORS[i]
    ax.add_patch(plt.Circle((x0, 0.5), 0.4, color=color, zorder=3))
    ax.add_patch(plt.Circle((x1, 0.5), 0.4, color=color, alpha=0.55, zorder=3))
    ax.text(x0, 0.5, la, ha='center', va='center', fontsize=13,
            fontweight='bold', color='white', zorder=4)
    ax.text(x1, 0.5, lb, ha='center', va='center', fontsize=13,
            fontweight='bold', color='white', zorder=4)
    ax.annotate('', xy=(x1 - 0.45, 0.5), xytext=(x0 + 0.45, 0.5),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text((x0 + x1) / 2, -0.2, f'Pair {i}  (items {2*i}→{2*i+1})',
            ha='center', fontsize=9, color=color)

plt.tight_layout()
plt.savefig(str(VIZ / 'pair_task_structure.png'), dpi=150, bbox_inches='tight')
plt.show()